# Document Augmentation through Question Generation

## Overview

This technique **enriches** each text fragment by generating related questions and storing them alongside the original text in the vector store. When a user query matches a generated question better than the raw text, retrieval improves.

| Standard RAG | Augmented RAG |
|---|---|
| Store: original chunks only | Store: original chunks **+ generated questions** |
| Query matches chunk text | Query matches a **similar question** → maps back to chunk |
| May miss semantically equivalent queries | Catches more query phrasings |

## How It Works

1. Split document into large "documents" (context windows) and small "fragments" (retrieval units)
2. For each document, use the LLM to generate ~40 questions that the document can answer
3. Store both fragments and questions in FAISS — each linked to its parent document text
4. At query time, retrieve the best match (often an augmented question)
5. Use the parent document text as context for answer generation

## Models Used

- **LLM**: `gemma3:4b` via Ollama (local)
- **Embeddings**: `mxbai-embed-large:335m` via Ollama (local)

---
## Step 0: Import Packages

In [1]:
import re
import fitz
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_ollama import ChatOllama
from langchain_ollama.embeddings import OllamaEmbeddings

---
## Step 1: Set Up LLM and Embedding Model

In [2]:
llm = ChatOllama(model="gemma3:4b", temperature=0)
embedding_model = OllamaEmbeddings(model="mxbai-embed-large:335m")

print("LLM and embedding model ready")

LLM and embedding model ready


---
## Step 2: Load the PDF as Text

In [3]:
path = "data/Understanding_Climate_Change.pdf"

doc = fitz.open(path)
content = ""
for page_num in range(len(doc)):
    content += doc[page_num].get_text()

print(f"Loaded {len(doc)} pages, {len(content)} characters")

Loaded 33 pages, 72561 characters


---
## Step 3: Define Chunking Parameters

We use a **two-level** split:

| Level | Purpose | Size |
|---|---|---|
| **Document** (large chunks) | Context window for the LLM to generate questions from, and for answer generation | ~4000 words |
| **Fragment** (small chunks) | Retrieval units stored in the vector store | ~128 words |

Questions are generated at the **document level** — the LLM sees a large chunk and generates questions about the whole thing.

In [4]:
DOCUMENT_MAX_TOKENS = 4000
DOCUMENT_OVERLAP_TOKENS = 100

FRAGMENT_MAX_TOKENS = 128
FRAGMENT_OVERLAP_TOKENS = 16

QUESTIONS_PER_DOCUMENT = 40

print(f"Documents: {DOCUMENT_MAX_TOKENS} words, {DOCUMENT_OVERLAP_TOKENS} overlap")
print(f"Fragments: {FRAGMENT_MAX_TOKENS} words, {FRAGMENT_OVERLAP_TOKENS} overlap")
print(f"Questions per document: {QUESTIONS_PER_DOCUMENT}")

Documents: 4000 words, 100 overlap
Fragments: 128 words, 16 overlap
Questions per document: 40


---
## Step 4: Split Text into Documents and Fragments

We split by **word count** (not character count) using a simple regex tokenizer.

In [5]:
# Split the full text into word-level tokens
all_tokens = re.findall(r'\b\w+\b', content)
print(f"Total words: {len(all_tokens)}")

# --- Split into large documents ---
text_documents = []
for i in range(0, len(all_tokens), DOCUMENT_MAX_TOKENS - DOCUMENT_OVERLAP_TOKENS):
    chunk_tokens = all_tokens[i:i + DOCUMENT_MAX_TOKENS]
    text_documents.append(" ".join(chunk_tokens))
    if i + DOCUMENT_MAX_TOKENS >= len(all_tokens):
        break

print(f"Split into {len(text_documents)} documents")

# --- Split each document into small fragments ---
all_fragments = []  # list of (fragment_text, parent_document_text)

for i, text_document in enumerate(text_documents):
    doc_tokens = re.findall(r'\b\w+\b', text_document)
    fragments = []
    for j in range(0, len(doc_tokens), FRAGMENT_MAX_TOKENS - FRAGMENT_OVERLAP_TOKENS):
        frag_tokens = doc_tokens[j:j + FRAGMENT_MAX_TOKENS]
        fragments.append(" ".join(frag_tokens))
        if j + FRAGMENT_MAX_TOKENS >= len(doc_tokens):
            break
    all_fragments.extend([(frag, text_document) for frag in fragments])
    print(f"  Document {i}: {len(fragments)} fragments")

print(f"\nTotal fragments: {len(all_fragments)}")

Total words: 9406
Split into 3 documents
  Document 0: 36 fragments
  Document 1: 36 fragments
  Document 2: 15 fragments

Total fragments: 87


---
## Step 5: Generate Questions for Each Document

This is the **core augmentation step**. For each large document chunk, we ask the LLM to generate questions that can be answered from it. These questions will be stored alongside the original fragments in the vector store.

In [6]:
question_schema = {
    "title": "QuestionList",
    "type": "object",
    "properties": {
        "question_list": {
            "type": "array",
            "items": {"type": "string"},
            "description": "List of questions generated for the document"
        }
    },
    "required": ["question_list"]
}

question_prompt = PromptTemplate(
    input_variables=["context", "num_questions"],
    template=(
        "Using the context data: {context}\n\n"
        "Generate a list of at least {num_questions} possible questions that can be asked about this context. "
        "Ensure the questions are directly answerable within the context and do not include any answers or headers. "
        "Separate the questions with a new line character."
    )
)

question_chain = question_prompt | llm.with_structured_output(question_schema)

print("Question generation chain ready")

Question generation chain ready


In [7]:
# Generate questions for each document and collect all augmented questions
all_questions = []  # list of (question_text, parent_document_text)

for i, text_document in enumerate(text_documents):
    result = question_chain.invoke({"context": text_document, "num_questions": QUESTIONS_PER_DOCUMENT})
    raw_questions = result["question_list"]

    # Clean: remove numbering, keep only lines ending with ?
    cleaned = []
    for q in raw_questions:
        q_clean = re.sub(r'^\d+\.\s*', '', q.strip())
        if q_clean.endswith('?'):
            cleaned.append(q_clean)
    cleaned = list(set(cleaned))  # deduplicate

    all_questions.extend([(q, text_document) for q in cleaned])
    print(f"  Document {i}: generated {len(cleaned)} questions")

print(f"\nTotal augmented questions: {len(all_questions)}")
print(f"\nSample questions:")
for q, _ in all_questions[:5]:
    print(f"  - {q}")

  Document 0: generated 53 questions
  Document 1: generated 16 questions
  Document 2: generated 28 questions

Total augmented questions: 97

Sample questions:
  - What is the role of carbon capture and storage technologies?
  - How can we foster innovation in climate technologies?
  - What is the role of youth engagement in addressing climate change?
  - What are the potential health impacts of climate change?
  - How does ocean acidification impact marine ecosystems?


---
## Step 6: Build the FAISS Vector Store

We store **both** original fragments and augmented questions. Each item's metadata includes:
- `type`: `"ORIGINAL"` or `"AUGMENTED"`
- `text`: the full parent document text (used as context for answer generation)

In [8]:
documents = []

# Add original fragments
for idx, (frag_text, parent_text) in enumerate(all_fragments):
    documents.append(Document(
        page_content=frag_text,
        metadata={"type": "ORIGINAL", "index": idx, "text": parent_text}
    ))

# Add augmented questions
for idx, (question_text, parent_text) in enumerate(all_questions):
    documents.append(Document(
        page_content=question_text,
        metadata={"type": "AUGMENTED", "index": len(all_fragments) + idx, "text": parent_text}
    ))

print(f"Total FAISS documents: {len(documents)}")
print(f"  Original fragments: {len(all_fragments)}")
print(f"  Augmented questions: {len(all_questions)}")

vectorstore = FAISS.from_documents(documents, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

print("\nVector store and retriever ready")

Total FAISS documents: 184
  Original fragments: 87
  Augmented questions: 97

Vector store and retriever ready


---
## Step 7: Test Retrieval — Direct Match

This query should match an original fragment directly.

In [9]:
query1 = "What is climate change?"
print(f"Query: {query1}\n")

results1 = retriever.invoke(query1)
for r in results1:
    print(f"Type: {r.metadata['type']}")
    print(f"Content: {r.page_content[:200]}...")

Query: What is climate change?

Type: ORIGINAL
Content: Understanding Climate Change Chapter 1 Introduction to Climate Change Climate change refers to significant long term changes in the global climate The term global climate encompasses the planet s over...


---
## Step 8: Test Retrieval — Augmented Question Match

This query is phrased differently from the original text. It should match an **augmented question** rather than the original fragment — this is the power of document augmentation.

In [10]:
query2 = "How do freshwater ecosystems change due to alterations in climatic factors?"
print(f"Query: {query2}\n")

results2 = retriever.invoke(query2)
for r in results2:
    print(f"Type: {r.metadata['type']}")
    print(f"Content: {r.page_content}")

print("\nNote: The retrieved item is likely an AUGMENTED question — the system matched")
print("our query against a pre-generated question that covers the same topic.")

Query: How do freshwater ecosystems change due to alterations in climatic factors?

Type: AUGMENTED
Content: How does climate change affect freshwater ecosystems?

Note: The retrieved item is likely an AUGMENTED question — the system matched
our query against a pre-generated question that covers the same topic.


---
## Step 9: Generate an Answer Using the Parent Document

The retrieved item (whether original or augmented) carries the full parent document in its metadata. We use that as context for answer generation.

In [11]:
# Get the parent document text from the retrieved item's metadata
context = results2[0].metadata["text"]
print(f"Context length: {len(context)} characters (from parent document)\n")

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant for question-answering tasks"),
    ("human", "Using the context data: {context}\n\nProvide a brief and precise answer to the question: {question}"),
])

answer_chain = answer_prompt | llm
answer = answer_chain.invoke({"context": context, "question": query2})

print(f"Question: {query2}")
print(f"\nAnswer: {answer.content}")

Context length: 28589 characters (from parent document)

Question: How do freshwater ecosystems change due to alterations in climatic factors?

Answer: Freshwater ecosystems change due to climate through altered water temperatures, altered precipitation patterns (leading to droughts or floods), and rising water levels, impacting species distribution, habitat availability, and overall ecosystem health.


---
## Summary

| Step | What happened |
|---|---|
| 1 | Set up LLM + embeddings |
| 2 | Loaded PDF as text |
| 3-4 | Split into large documents (4000 words) and small fragments (128 words) |
| 5 | **Generated ~40 questions per document** using the LLM |
| 6 | Built FAISS store with **both** original fragments and augmented questions |
| 7 | Tested with a direct query → matched an original fragment |
| 8 | Tested with a rephrased query → matched an **augmented question** |
| 9 | Used the parent document as context to generate an answer |

**Key insight:** By storing LLM-generated questions alongside the original text, the vector store captures many more ways a topic might be queried. When a user asks "How do freshwater ecosystems change due to climatic factors?" — the exact text may not contain those words, but a generated question like "How does climate change affect freshwater ecosystems?" is a close match. The augmented question then links back to the full parent document for answer generation.